# 03 – Baseline models

**Goal:** Build simple first models. Their scores are the standard that XGBoost (Day 2) must beat.

**Baseline** = a simple first model. Without it, an error like "$48,000" means nothing, because there's nothing to compare it with.

**Data:** all 8 features → `MedHouseVal` (in 100,000 USD). Same split as notebooks 01 and 02 (80/20, random_state=42).

In [1]:
import sklearn
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

print("scikit-learn:", sklearn.__version__)

housing_df = fetch_california_housing(as_frame=True).frame

feature_columns = housing_df.columns.drop("MedHouseVal")
X = housing_df[feature_columns]
y = housing_df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Features:", list(feature_columns))
print("Train:", X_train.shape, " Test:", X_test.shape)

scikit-learn: 1.9.1
Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train: (16512, 8)  Test: (4128, 8)


## Regression baseline: StandardScaler + LinearRegression

**Pipeline** = scaler → model in one object. `.fit()` scales and trains; `.predict()` scales new data the same way.

**StandardScaler:** `scaled = (value − average) / spread`. 0 = average. Learned from the **train set only**, so there's no leakage.

We also measure the **training time**, which XGBoost will be compared on later.

In [2]:
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

baseline_regression_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])

start_time = time.perf_counter()
baseline_regression_pipeline.fit(X_train, y_train)
regression_train_time_seconds = time.perf_counter() - start_time

fitted_linear_model = baseline_regression_pipeline.named_steps["model"]
print(f"Training time: {regression_train_time_seconds:.3f} seconds")
print("Bias (b):", round(fitted_linear_model.intercept_, 3))
print(pd.Series(fitted_linear_model.coef_, index=feature_columns).round(3))

Training time: 0.062 seconds
Bias (b): 2.072
MedInc        0.854
HouseAge      0.123
AveRooms     -0.294
AveBedrms     0.339
Population   -0.002
AveOccup     -0.041
Latitude     -0.897
Longitude    -0.870
dtype: float64
